# Wilderness Outfitters: the Christmas stock plan

This is the finished version of Lab 2. It builds Wilderness's Christmas plan from the data loaded at setup, in the same order as the lab, and shows each result as it goes.

**Today is Wednesday 4 November 2026.** The data runs to Tuesday 3 November.

| Part | Question | What it creates |
|---|---|---|
| 1 | Does each state have enough tents? | Nothing; it reads the data |
| 2 | Is demand up or down? | Nothing; it reads the data |
| 3 | Why are Summit Tents coming back? | `TENT_RETURN_REASONS`, labelled by `AI_CLASSIFY` |
| 4 | Who bought a tent without a sleeping bag? | `SLEEPING_BAG_OFFER`, with five emails written by `AI_COMPLETE` |
| 5 | What does each store do? | `CHRISTMAS_PLAN`, a dynamic table that keeps itself up to date |

**How to use it:** select **Run all**, or run the cells one at a time from the top. Running it again replaces what it built.

## Before you start

Point the session at the workshop database, warehouse and schema. Everything this notebook creates goes in `LAB2_ANALYTICS`.

In [ ]:
-- Point the session at the workshop
USE ROLE ACCOUNTADMIN;
USE WAREHOUSE WILDERNESS_WH;
USE DATABASE WILDERNESS_HOL;
USE SCHEMA LAB2_ANALYTICS;

## 1. Does each state have enough tents?

Wilderness bought 40% more tents for every store. This compares each state's tents on hand at 3 November with the tents it sold over the same weeks last year, 4 November to 24 December.

In [ ]:
WITH tents AS (SELECT PRODUCT_ID FROM WILDERNESS_HOL.RAW.DIM_PRODUCT WHERE CATEGORY = 'Tents & Shelters'),
on_hand AS (
  SELECT s.STATE, SUM(i.ON_HAND_UNITS) AS TENTS_ON_HAND
  FROM WILDERNESS_HOL.RAW.FCT_INVENTORY i
  JOIN tents t ON t.PRODUCT_ID = i.PRODUCT_ID
  JOIN WILDERNESS_HOL.RAW.DIM_STORE s ON s.STORE_ID = i.STORE_ID
  GROUP BY 1),
sold AS (
  SELECT s.STATE, SUM(f.QUANTITY) AS TENTS_SOLD_LY
  FROM WILDERNESS_HOL.RAW.FCT_SALES_LINE f
  JOIN tents t ON t.PRODUCT_ID = f.PRODUCT_ID
  JOIN WILDERNESS_HOL.RAW.DIM_STORE s ON s.STORE_ID = f.STORE_ID
  WHERE f.ORDER_DATE BETWEEN '2025-11-04' AND '2025-12-24'
  GROUP BY 1)
SELECT o.STATE, o.TENTS_ON_HAND, s.TENTS_SOLD_LY, ROUND(o.TENTS_ON_HAND / s.TENTS_SOLD_LY, 2) AS STOCK_VS_LAST_YEAR
FROM on_hand o JOIN sold s ON s.STATE = o.STATE
ORDER BY 1;

**What you should see:** `STOCK_VS_LAST_YEAR` is about 1.4 in every state. The extra tents went everywhere.

## 2. Is demand up or down?

The first three days of Christmas trading, 1 to 3 November, this year against the same days last year. Switch the result to **Chart** to see it as a bar chart.

In [ ]:
SELECT s.STATE,
       SUM(IFF(f.ORDER_DATE BETWEEN '2025-11-01' AND '2025-11-03', f.QUANTITY, 0)) AS TENTS_2025,
       SUM(IFF(f.ORDER_DATE BETWEEN '2026-11-01' AND '2026-11-03', f.QUANTITY, 0)) AS TENTS_2026,
       ROUND(TENTS_2026 / TENTS_2025 - 1, 3) AS GROWTH
FROM WILDERNESS_HOL.RAW.FCT_SALES_LINE f
JOIN WILDERNESS_HOL.RAW.DIM_PRODUCT p ON p.PRODUCT_ID = f.PRODUCT_ID
JOIN WILDERNESS_HOL.RAW.DIM_STORE s ON s.STORE_ID = f.STORE_ID
WHERE p.CATEGORY = 'Tents & Shelters'
  AND (f.ORDER_DATE BETWEEN '2025-11-01' AND '2025-11-03' OR f.ORDER_DATE BETWEEN '2026-11-01' AND '2026-11-03')
GROUP BY 1
ORDER BY GROWTH DESC;

**What you should see:** two groups. New South Wales and Queensland are up about 60%; every other state is up about 10%. So those two states, with 1.4 times the stock against 1.6 times the demand, run out before Christmas Eve, and the rest have tents left over.

## 3. Why are Summit Tents coming back?

### The return rate by batch

How many Summit Tents from each batch were sold, and how many came back.

In [ ]:
SELECT f.BATCH_CODE, SUM(f.QUANTITY) AS SOLD, SUM(COALESCE(r.RETURN_QUANTITY, 0)) AS RETURNED,
       ROUND(RETURNED / SOLD, 3) AS RETURN_RATE
FROM WILDERNESS_HOL.RAW.FCT_SALES_LINE f
LEFT JOIN WILDERNESS_HOL.RAW.FCT_RETURN r ON r.SALE_LINE_ID = f.SALE_LINE_ID
WHERE f.PRODUCT_ID = 'SKU-TNT-0102' AND f.BATCH_CODE IS NOT NULL
GROUP BY 1
ORDER BY RETURN_RATE DESC;

**What you should see:** batch `B26-07` comes back at about 18%; every other batch at about 4%.

### Label every return comment with AI_CLASSIFY

The returns desk codes nearly every return as faulty, so the codes cannot say which returns are real faults. The customer's comment can: a snapped pole or a leaking seam is a fault; a first-time camper who could not put the tent up is not. `AI_CLASSIFY` reads every Summit Tent return comment since 1 July and gives each one of five labels. It runs in SQL, on every row, and saves the result as a table.

In [ ]:
CREATE OR REPLACE TABLE WILDERNESS_HOL.LAB2_ANALYTICS.TENT_RETURN_REASONS AS
SELECT FEEDBACK_ID, BATCH_CODE, FEEDBACK_DATE,
       AI_CLASSIFY(FEEDBACK_TEXT, ['POLE_FAULT', 'SEAM_FAULT', 'COULD_NOT_PITCH', 'CHANGE_OF_MIND', 'OTHER']):labels[0]::VARCHAR AS LABEL
FROM WILDERNESS_HOL.RAW.CUSTOMER_FEEDBACK
WHERE PRODUCT_ID = 'SKU-TNT-0102'
  AND FEEDBACK_SOURCE = 'RETURN_COMMENT'
  AND FEEDBACK_DATE >= '2026-07-01';

### Count the labels by batch

In [ ]:
SELECT BATCH_CODE, LABEL, COUNT(*) AS COMMENTS,
       ROUND(RATIO_TO_REPORT(COUNT(*)) OVER (PARTITION BY BATCH_CODE), 3) AS SHARE_OF_BATCH
FROM WILDERNESS_HOL.LAB2_ANALYTICS.TENT_RETURN_REASONS
GROUP BY 1, 2
ORDER BY 1, 3 DESC;

**What you should see:** seven in ten `B26-07` returns are `POLE_FAULT` or `SEAM_FAULT`. About one in six are `COULD_NOT_PITCH`: first-time campers, not faults. The other batches' returns are mostly `CHANGE_OF_MIND`.

## 4. Who bought a tent without a sleeping bag?

### Find the Basecamp members

Members who bought a tent from 1 to 3 November with no sleeping bag or mat in the same order, one row each, with their first name, store and tent.

In [ ]:
CREATE OR REPLACE TABLE WILDERNESS_HOL.LAB2_ANALYTICS.SLEEPING_BAG_OFFER AS
WITH tent_orders AS (
  SELECT f.ORDER_ID, f.ORDER_DATE, f.MEMBER_ID, f.STORE_ID,
         MAX(IFF(p.CATEGORY = 'Tents & Shelters', p.PRODUCT_NAME, NULL)) AS TENT_NAME,
         MAX(IFF(p.KIT_GROUP = 'SLEEP', 1, 0)) AS HAS_SLEEP
  FROM WILDERNESS_HOL.RAW.FCT_SALES_LINE f
  JOIN WILDERNESS_HOL.RAW.DIM_PRODUCT p ON p.PRODUCT_ID = f.PRODUCT_ID
  WHERE f.ORDER_DATE BETWEEN '2026-11-01' AND '2026-11-03' AND f.MEMBER_ID IS NOT NULL
  GROUP BY 1, 2, 3, 4)
SELECT m.MEMBER_ID, m.FIRST_NAME, s.STORE_NAME, t.TENT_NAME, NULL::VARCHAR AS EMAIL
FROM tent_orders t
JOIN WILDERNESS_HOL.RAW.DIM_MEMBER m ON m.MEMBER_ID = t.MEMBER_ID
JOIN WILDERNESS_HOL.RAW.DIM_STORE s ON s.STORE_ID = t.STORE_ID
WHERE t.TENT_NAME IS NOT NULL AND t.HAS_SLEEP = 0
QUALIFY ROW_NUMBER() OVER (PARTITION BY m.MEMBER_ID ORDER BY t.ORDER_DATE, t.ORDER_ID) = 1;

**What you should see:** about 700 members in `SLEEPING_BAG_OFFER`.

### Write five offer emails with AI_COMPLETE

`AI_COMPLETE` writes text in SQL. Each member's name, tent and store go into the prompt, and the email comes back in the `EMAIL` column. It is charged by the amount of text, so it writes five for review before anyone writes 700. Nothing here sends an email.

In [ ]:
UPDATE WILDERNESS_HOL.LAB2_ANALYTICS.SLEEPING_BAG_OFFER
SET EMAIL = AI_COMPLETE('claude-sonnet-4-6',
  'Write a short email from Wilderness Outfitters, an Australian outdoor retailer, to a Basecamp loyalty member. '
  || 'They are ' || FIRST_NAME || ', who bought a ' || TENT_NAME || ' at ' || STORE_NAME
  || ' this week but no sleeping bag or mat. Thank them for their tent, and offer 15% off a sleeping bag or mat '
  || 'before Christmas. Plain Australian English, under 80 words, no exclamation marks, no emoji. '
  || 'Return only a subject line and the body.')::VARCHAR
WHERE MEMBER_ID IN (SELECT MEMBER_ID FROM WILDERNESS_HOL.LAB2_ANALYTICS.SLEEPING_BAG_OFFER ORDER BY MEMBER_ID LIMIT 5);

### Read the emails

In [ ]:
SELECT FIRST_NAME, TENT_NAME, STORE_NAME, EMAIL
FROM WILDERNESS_HOL.LAB2_ANALYTICS.SLEEPING_BAG_OFFER
WHERE EMAIL IS NOT NULL
ORDER BY MEMBER_ID;

**What you should see:** five short emails, each naming the member, their tent and their store. The wording changes from run to run.

## 5. What does each store do?

### Build the plan as a dynamic table

One row per store. The faulty `B26-07` tents go back to Kestrel, the supplier, so they are taken off first. The forecast to Christmas Eve is last year's tent sales from 4 November to 24 December, grown by the state's growth so far. A store with more good tents than its forecast moves the spare out (`MOVE OUT`); a store with fewer needs tents moved in (`MOVE IN`). Snowflake refreshes the table within an hour of new data, with no job to schedule.

In [ ]:
CREATE OR REPLACE DYNAMIC TABLE WILDERNESS_HOL.LAB2_ANALYTICS.CHRISTMAS_PLAN
  TARGET_LAG = '1 hour'
  WAREHOUSE = WILDERNESS_WH
  COMMENT = 'Christmas plan at 3 Nov 2026: one row per store, tents to send back and move'
AS
WITH tents AS (
  SELECT PRODUCT_ID FROM WILDERNESS_HOL.RAW.DIM_PRODUCT WHERE CATEGORY = 'Tents & Shelters'),
growth AS (
  SELECT s.STATE,
         SUM(IFF(f.ORDER_DATE BETWEEN '2026-11-01' AND '2026-11-03', f.QUANTITY, 0))
         / NULLIF(SUM(IFF(f.ORDER_DATE BETWEEN '2025-11-01' AND '2025-11-03', f.QUANTITY, 0)), 0) AS GROWTH
  FROM WILDERNESS_HOL.RAW.FCT_SALES_LINE f
  JOIN tents t ON t.PRODUCT_ID = f.PRODUCT_ID
  JOIN WILDERNESS_HOL.RAW.DIM_STORE s ON s.STORE_ID = f.STORE_ID
  WHERE f.ORDER_DATE BETWEEN '2025-11-01' AND '2025-11-03' OR f.ORDER_DATE BETWEEN '2026-11-01' AND '2026-11-03'
  GROUP BY 1),
last_year AS (
  SELECT f.STORE_ID, SUM(f.QUANTITY) AS TENTS_SOLD_LY
  FROM WILDERNESS_HOL.RAW.FCT_SALES_LINE f
  JOIN tents t ON t.PRODUCT_ID = f.PRODUCT_ID
  WHERE f.ORDER_DATE BETWEEN '2025-11-04' AND '2025-12-24'
  GROUP BY 1),
stock AS (
  SELECT i.STORE_ID, SUM(i.ON_HAND_UNITS) AS TENTS_ON_HAND,
         SUM(IFF(i.BATCH_CODE = 'B26-07', i.ON_HAND_UNITS, 0)) AS TENTS_TO_SEND_BACK
  FROM WILDERNESS_HOL.RAW.FCT_INVENTORY i
  JOIN tents t ON t.PRODUCT_ID = i.PRODUCT_ID
  GROUP BY 1),
plan AS (
  SELECT s.STORE_ID, s.STORE_NAME, s.STATE, s.LATITUDE, s.LONGITUDE,
         st.TENTS_ON_HAND, st.TENTS_TO_SEND_BACK,
         ROUND(ly.TENTS_SOLD_LY * g.GROWTH) AS FORECAST_TO_XMAS
  FROM WILDERNESS_HOL.RAW.DIM_STORE s
  JOIN stock st ON st.STORE_ID = s.STORE_ID
  JOIN last_year ly ON ly.STORE_ID = s.STORE_ID
  JOIN growth g ON g.STATE = s.STATE)
SELECT STORE_ID, STORE_NAME, STATE, LATITUDE, LONGITUDE, TENTS_ON_HAND, TENTS_TO_SEND_BACK, FORECAST_TO_XMAS,
       TENTS_ON_HAND - TENTS_TO_SEND_BACK - FORECAST_TO_XMAS AS GAP,
       IFF(TENTS_ON_HAND - TENTS_TO_SEND_BACK - FORECAST_TO_XMAS > 0, 'MOVE OUT', 'MOVE IN') AS ACTION
FROM plan;

### The plan in four numbers

In [ ]:
SELECT SUM(TENTS_TO_SEND_BACK) AS TENTS_TO_SEND_BACK,
       SUM(IFF(ACTION = 'MOVE OUT', GAP, 0)) AS TENTS_SPARE,
       -SUM(IFF(ACTION = 'MOVE IN', GAP, 0)) AS TENTS_SHORT,
       TENTS_SHORT - TENTS_SPARE AS TENTS_TO_REORDER,
       COUNT_IF(ACTION = 'MOVE OUT') AS STORES_MOVE_OUT,
       COUNT_IF(ACTION = 'MOVE IN') AS STORES_MOVE_IN
FROM WILDERNESS_HOL.LAB2_ANALYTICS.CHRISTMAS_PLAN;

**What you should see:** about 5,100 tents to send back, about 2,400 spare and about 5,300 short. The spare does not cover the shortfall, so about 2,900 tents are reordered by 13 November. 39 stores move tents out and 33 move them in.

## What is next

The merchandising team opens the plan in the `CHRISTMAS_PLANNER` app: in Snowsight, go to **Projects » Streamlit**. If you did not build it in the lab, the catch-up script `91_catch_up_lab2.sql` deploys the finished one.